# Analyse des sentiments en langue arabe par fine-tuning d’AraBERT

**Objectif :** adapter `aubmindlab/bert-base-arabertv02` à la classification binaire de tweets
arabes : **négatif** ou **positif**. Le corpus ne contient pas de classe neutre.

Le **préentraînement** apprend des régularités de la langue sur de très nombreux textes,
notamment en prédisant des mots masqués. Le **fine-tuning** ajoute une tête de classification
et ajuste les poids du réseau avec des tweets et leurs sentiments annotés.

Exécuter les cellules dans l’ordre. Le mode rapide est activé par défaut : il vérifie la chaîne
complète avec le vrai AraBERT sur un petit échantillon. Ses scores ne constituent pas un bilan
du corpus complet. Passer ensuite `QUICK_MODE = False` et redémarrer le noyau pour une expérience complète.

Sources : [corpus Kaggle](https://www.kaggle.com/datasets/mksaad/arabic-sentiment-twitter-corpus),
[dépôt de l’auteur du corpus](https://github.com/motazsaad/arabic-sentiment-analysis),
[modèle](https://huggingface.co/aubmindlab/bert-base-arabertv02),
[prétraitement officiel](https://github.com/aub-mind/arabert/blob/master/preprocess.py),
[Trainer 4.51.3](https://huggingface.co/docs/transformers/v4.51.3/en/main_classes/trainer).

Ce notebook est fourni sans résultats d’entraînement préremplis. La dernière cellule calcule
la conclusion à partir des sorties de l’exécution en cours.


In [ ]:
import sys
import subprocess

if not ((3, 10) <= sys.version_info[:2] <= (3, 12)):
    raise RuntimeError("Cette pile est prévue pour Python 3.10–3.12. Sélectionnez un noyau compatible.")
INSTALL_DEPENDENCIES = True  # False si requirements.txt est déjà installé dans ce noyau.
PACKAGES = ['torch>=2.6.0,<2.9', 'transformers==4.51.3', 'tokenizers==0.21.1', 'huggingface-hub==0.30.2', 'safetensors==0.5.3', 'datasets==3.5.0', 'accelerate==1.6.0', 'arabert==1.0.1', 'emoji==1.4.2', 'numpy==1.26.4', 'pandas==2.2.3', 'scikit-learn==1.6.1', 'matplotlib==3.10.1', 'seaborn==0.13.2', 'pyarrow==19.0.1', 'fsspec==2024.12.0', 'kaggle==1.7.4.5', 'nbformat==5.10.4', 'nbclient==0.10.2', 'ipykernel==6.29.5', 'jupyterlab==4.4.0']
if INSTALL_DEPENDENCIES:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *PACKAGES])
print("Python :", sys.version.split()[0])
print("Installation terminée. Si des bibliothèques étaient déjà importées, redémarrer le noyau.")


## 1. Imports, matériel et reproductibilité

Sur **Kaggle**, activer Internet et choisir un accélérateur GPU. Sur **Colab**, choisir un
environnement GPU et importer le notebook. En **local**, utiliser un environnement Python
3.10–3.12 ; pour NVIDIA, installer une roue PyTorch adaptée au pilote depuis
[le sélecteur officiel](https://pytorch.org/get-started/locally/) avant les dépendances.
Le CPU reste utilisable, mais le fine-tuning complet peut être très long.

Les graines et les réglages ci-dessous améliorent la reproductibilité sans garantir une identité
parfaite entre GPU, pilotes et systèmes. Aucun service de suivi externe n’est nécessaire.


In [ ]:
import os
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import csv
import gc
import hashlib
import importlib.metadata as metadata
import json
import math
import platform
import random
import re
import unicodedata
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix)
from datasets import Dataset, DatasetDict
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, Trainer, TrainingArguments, set_seed)
from arabert.preprocess import ArabertPreprocessor

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)
sns.set_theme(style="whitegrid", palette="colorblind")
pd.set_option("display.max_colwidth", 180)

CUDA = torch.cuda.is_available()
MPS = bool(hasattr(torch.backends, "mps") and torch.backends.mps.is_available())
DEVICE = torch.device("cuda" if CUDA else "mps" if MPS else "cpu")
print("Système :", platform.platform(), "| Python :", platform.python_version())
print("PyTorch :", torch.__version__, "| CUDA de la roue :", torch.version.cuda)
print("Périphérique :", DEVICE)
if CUDA:
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"GPU {i} : {props.name} | {props.total_memory / 2**30:.1f} Gio | capacité {props.major}.{props.minor}")
else:
    print("GPU CUDA indisponible ; précision float32 sur", DEVICE)
VERSIONS = {name: metadata.version(name) for name in
            ("torch", "transformers", "datasets", "accelerate", "arabert", "emoji",
             "numpy", "pandas", "scikit-learn", "matplotlib", "seaborn", "tokenizers")}
display(pd.Series(VERSIONS, name="version").to_frame())
assert VERSIONS["transformers"] == "4.51.3", "Réexécuter l’installation puis redémarrer le noyau."


## 2. Configuration de l’expérience

`DATA_DIR` est le seul chemin à adapter pour un import manuel. La recherche automatique
reconnaît aussi le dossier du corpus sous `/kaggle/input`. Les sorties vont dans un dossier
distinct pour chaque exécution afin de ne pas réutiliser un ancien checkpoint par accident.

Le corpus fourni possède un test officiel : `EXPECT_OFFICIAL_TEST` doit rester à `True`.
La branche sans test officiel est disponible pour une autre distribution effectivement dépourvue
de test : mettre alors explicitement cette option à `False`. Un test manquant dans une copie
incomplète du corpus déclenche une erreur, pas un nouveau découpage silencieux.


In [ ]:
MODEL_NAME = "aubmindlab/bert-base-arabertv02"
MODEL_REVISION = "016fb9d6768f522a59c6e0d2d5d5d43a4e1bff60"
DATA_DIR = Path("./data")  # Exemple Colab : Path("/content/data") ; chemin absolu accepté.
DOWNLOAD_WITH_KAGGLE = False
EXPECT_OFFICIAL_TEST = True
QUICK_MODE = True
QUICK_LIMITS = {"train": 256, "validation": 128, "test": 128}
VAL_FRACTION = 0.10  # Part du train officiel ; si aucun test : 80/10/10.
TEST_FRACTION = 0.10
LENGTH_QUANTILE = 0.95
MEMORY_LENGTH_CAP = 256  # Plafond de mémoire, distinct de la limite du modèle.
EPOCHS = 1 if QUICK_MODE else 3
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
OUTPUT_ROOT = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
RUN_DIR = OUTPUT_ROOT / "outputs" / (f"quick_{RUN_ID}" if QUICK_MODE else f"full_{RUN_ID}")
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("Mode :", "RAPIDE, diagnostic uniquement" if QUICK_MODE else "COMPLET")
print("Sorties :", RUN_DIR.resolve())


## 3. Obtenir le corpus : API Kaggle ou import manuel

**API Kaggle :** configurer l’authentification à l’extérieur du notebook avec le mécanisme
officiel de Kaggle (fichier privé de configuration ou variables d’environnement gérées par
votre environnement), puis activer `DOWNLOAD_WITH_KAGGLE`. Aucun identifiant n’est écrit
ou affiché ici. Sur Kaggle, ajouter directement le dataset à l’entrée du notebook évite cette étape.

**Import manuel :** télécharger l’archive depuis la page Kaggle et l’extraire dans `DATA_DIR`.
Sur Colab, l’envoyer avec le panneau **Fichiers** ou monter Drive puis renseigner son chemin.
Fournir les **quatre TSV** ; ne pas mélanger les fichiers individuels de `arabic_tweets/`
avec les TSV. Les deux représentations du même corpus figurent dans l’archive inspectée.


In [ ]:
if DOWNLOAD_WITH_KAGGLE:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    try:
        from kaggle.api.kaggle_api_extended import KaggleApi
        kaggle_api = KaggleApi()
        kaggle_api.authenticate()
        kaggle_api.dataset_download_files(
            "mksaad/arabic-sentiment-twitter-corpus",
            path=str(DATA_DIR), unzip=True, quiet=True,
        )
        print("Téléchargement Kaggle terminé.")
    except Exception:
        raise RuntimeError(
            "Téléchargement impossible. Configurer Kaggle hors du notebook, ou extraire "
            "manuellement les quatre TSV dans DATA_DIR puis désactiver DOWNLOAD_WITH_KAGGLE."
        ) from None
else:
    print("Import manuel ou dataset Kaggle déjà attaché : recherche des fichiers à l’étape suivante.")


## 4. Inspecter les vrais fichiers avant de lire un tableau

L’archive accessible lors de la préparation a été inspectée :

| Fichier | Lignes observées |
|---|---:|
| `train_Arabic_tweets_negative_20190413.tsv` | 23 121 |
| `train_Arabic_tweets_positive_20190413.tsv` | 23 879 |
| `test_Arabic_tweets_negative_20190413.tsv` | 5 781 |
| `test_Arabic_tweets_positive_20190413.tsv` | 5 970 |

Ces quatre fichiers sont décodables strictement en **UTF-8**, sans en-tête, avec **deux champs
séparés par une tabulation** : `label`, puis `tweet`. Les valeurs sont `neg` et `pos`.
Les guillemets appartiennent au texte. La cellule suivante revérifie chaque ligne et affiche
des extraits bruts, les effectifs, les labels et les empreintes SHA-256 **avant** de choisir
`read_csv`. Un format différent interrompt la lecture avec un diagnostic.


In [ ]:
TRAIN_NAMES = ["train_Arabic_tweets_negative_20190413.tsv", "train_Arabic_tweets_positive_20190413.tsv"]
TEST_NAMES = ["test_Arabic_tweets_negative_20190413.tsv", "test_Arabic_tweets_positive_20190413.tsv"]
search_roots = [DATA_DIR, Path.cwd()]
if Path("/kaggle/input").is_dir():
    search_roots.append(Path("/kaggle/input"))

file_paths = {}
for name in TRAIN_NAMES + TEST_NAMES:
    for root in search_roots:
        if root.is_dir():
            direct = root / name
            # Ne pas parcourir les sorties/venv locaux : recherche récursive dans les entrées seulement.
            matches = [direct] if direct.is_file() else (list(root.rglob(name)) if root != Path.cwd() else [])
            if len(matches) > 1:
                raise RuntimeError(f"Plusieurs copies de {name}. Choisir un DATA_DIR plus précis.")
            if matches:
                file_paths[name] = matches[0].resolve()
                break
missing_train = sorted(set(TRAIN_NAMES) - set(file_paths))
present_test = [name for name in TEST_NAMES if name in file_paths]
if missing_train or (EXPECT_OFFICIAL_TEST and len(present_test) != 2) or len(present_test) == 1:
    raise FileNotFoundError(
        f"Copie absente/incomplète dans {DATA_DIR.resolve()}. Fournir les quatre TSV de Kaggle. "
        f"Fichiers trouvés : {list(file_paths)}. Revoir DATA_DIR puis réexécuter cette cellule."
    )
HAS_OFFICIAL_TEST = len(present_test) == 2
inspection = []
for name, path in file_paths.items():
    raw = path.read_bytes()
    try:
        decoded = raw.decode("utf-8-sig", errors="strict")
    except UnicodeDecodeError:
        raise ValueError(f"{name} n’est pas UTF-8 : vérifier le fichier source et son encodage.") from None
    lines = decoded.splitlines()
    bad_lines = [i for i, line in enumerate(lines, start=1) if len(line.split("\t")) != 2]
    if bad_lines or not lines:
        raise ValueError(f"{name} : format différent du TSV à 2 champs ; lignes concernées : {bad_lines[:8]}")
    observed_labels = Counter(line.split("\t", 1)[0] for line in lines)
    expected_raw_label = "neg" if "negative" in name else "pos"
    if set(observed_labels) != {expected_raw_label}:
        raise ValueError(f"{name} : labels inattendus {observed_labels}. Consulter la documentation source.")
    inspection.append({"fichier": name, "chemin": str(path), "encodage": "UTF-8 (BOM accepté)",
                       "séparateur": "tabulation", "en_tête": False, "champs": "label, tweet",
                       "lignes": len(lines), "labels": dict(observed_labels),
                       "sha256": hashlib.sha256(raw).hexdigest()})
    print(name, "| extrait brut :", repr(lines[0][:180]))
inspection_df = pd.DataFrame(inspection)
display(inspection_df.drop(columns=["chemin", "sha256"]))
inspection_df.to_json(RUN_DIR / "data_inspection.json", orient="records", force_ascii=False, indent=2)


## 5. Lecture et exploration descriptive

Les noms de colonnes ci-dessous sont attribués explicitement car le corpus n’a pas d’en-tête.
La lecture ne saute aucune ligne mal formée. Les tableaux descriptifs montrent la composition
des données ; aucune performance sur test n’est calculée à ce stade.


In [ ]:
frames = []
for name, path in file_paths.items():
    frame = pd.read_csv(path, sep="\t", header=None, names=["raw_label", "text"],
                        encoding="utf-8-sig", dtype="string", quoting=csv.QUOTE_NONE,
                        keep_default_na=False, skip_blank_lines=False, on_bad_lines="error")
    frame["source_file"] = name
    frame["source_line"] = np.arange(1, len(frame) + 1)
    frame["official_split"] = "test" if name.startswith("test_") else "train"
    frame["example_id"] = frame["source_file"] + ":" + frame["source_line"].astype(str)
    frames.append(frame)
raw_df = pd.concat(frames, ignore_index=True)
print("Dimensions :", raw_df.shape)
display(raw_df.sample(min(6, len(raw_df)), random_state=SEED))
display(raw_df.isna().sum().rename("valeurs_manquantes").to_frame())
print("Champs texte vides :", int(raw_df["text"].str.strip().eq("").sum()))
print("Labels vides :", int(raw_df["raw_label"].str.strip().eq("").sum()))
raw_distribution = pd.crosstab(raw_df["official_split"], raw_df["raw_label"])
display(raw_distribution)
raw_distribution.plot(kind="bar", rot=0, figsize=(7, 3), title="Classes avant nettoyage")
plt.ylabel("Nombre de tweets")
plt.tight_layout()
plt.show()


## 6. Signification des labels et prétraitement AraBERT

La documentation de l’auteur décrit deux polarités et les noms des fichiers confirment
`negative → neg` et `positive → pos`. On construit donc **0 = négatif**, **1 = positif**,
plutôt que d’inférer le sens d’un nombre arbitraire.

Le modèle **v0.2 (`v02`) ne demande pas de segmentation Farasa**. On utilise le préprocesseur
officiel avec `keep_emojis=True` et sans réduction des répétitions expressives. Il normalise
notamment les espaces et remplace liens et mentions. On ne retire ni négations, ni mots outils,
ni hashtags par une règle supplémentaire ; aucune racinisation n’est appliquée.

Ce modèle de base n’a pas le vocabulaire emoji enrichi de la variante Twitter : conserver un
emoji dans le texte ne garantit pas qu’il soit représenté par un token distinct de `[UNK]`.
La cellule de tokenisation rend cette limite visible.


In [ ]:
RAW_LABEL_TO_ID = {"neg": 0, "pos": 1}
id2label = {0: "négatif", 1: "positif"}
label2id = {label: idx for idx, label in id2label.items()}
LABEL_IDS = [0, 1]
LABEL_NAMES = [id2label[i] for i in LABEL_IDS]
observed = set(raw_df["raw_label"].dropna().unique())
if observed != set(RAW_LABEL_TO_ID):
    raise ValueError(f"Labels observés : {observed} ; la correspondance doit être vérifiée dans la source.")
PREPROCESS_KWARGS = {"model_name": MODEL_NAME, "keep_emojis": True,
                     "apply_farasa_segmentation": False, "remove_non_digit_repetition": False}
arabert_preprocessor = ArabertPreprocessor(**PREPROCESS_KWARGS)

def preprocess_text(text):
    if not isinstance(text, str):
        raise TypeError("Le texte doit être une chaîne Unicode.")
    return arabert_preprocessor.preprocess(text).strip()

def text_fingerprint(text):
    canonical = " ".join(unicodedata.normalize("NFKC", text).split())
    return hashlib.sha256(canonical.encode("utf-8")).hexdigest()

assert "لا" in preprocess_text("لا أحب هذا أبداً 😞").split()
assert "لن" in preprocess_text("لن أكرر هذه التجربة").split()
assert "ليس" in preprocess_text("هذا ليس جيداً").split()
assert "😊" in preprocess_text("أنا سعيد 😊")
assert "😞" in preprocess_text("أنا حزين 😞")
print("Mapping :", id2label, "| inverse :", label2id)
print(preprocess_text("لا أحب هذه الخدمة!!! 😞 https://example.com @someone"))


## 7. Nettoyage, contradictions et protection du test

Un tweet vide est supprimé. Pour les doublons, on compare les textes après prétraitement et
normalisation Unicode. **Priorité au test officiel :** les copies présentes dans le train
sont retirées du train par comparaison des textes, sans consulter les labels du test.

À l’intérieur de chaque ensemble, un groupe de textes identiques avec des labels contradictoires
est exclu entièrement ; un doublon cohérent n’est gardé qu’une fois. Le test reste constitué
exclusivement de lignes du test officiel, avec ce nettoyage explicite et audité. Aucun de ses
exemples ne devient un exemple d’entraînement. Le journal mesure l’impact de chaque règle.


In [ ]:
cleaning_log = []

def audit(stage, before, after):
    cleaning_log.append({"étape": stage, "avant": int(before), "après": int(after),
                         "retirés": int(before - after)})

df = raw_df.dropna(subset=["text", "raw_label"]).copy()
df = df.loc[df["text"].str.strip().ne("")].copy()
audit("textes/labels manquants ou vides", len(raw_df), len(df))
df["labels"] = df["raw_label"].map(RAW_LABEL_TO_ID).astype("int64")
df["processed_text"] = df["text"].map(preprocess_text)
before = len(df)
df = df.loc[df["processed_text"].str.strip().ne("")].copy()
audit("vides après prétraitement", before, len(df))
df["text_key"] = df["processed_text"].map(text_fingerprint)

def clean_within(frame, key, name):
    frame = frame.sort_values("example_id", kind="stable").copy()
    label_counts = frame.groupby(key, sort=False)["labels"].nunique()
    conflict_keys = set(label_counts[label_counts > 1].index)
    before = len(frame)
    frame = frame.loc[~frame[key].isin(conflict_keys)].copy()
    audit(f"{name} : contradictions ({len(conflict_keys)} groupes)", before, len(frame))
    before = len(frame)
    frame = frame.drop_duplicates(key, keep="first").reset_index(drop=True)
    audit(f"{name} : doublons cohérents", before, len(frame))
    return frame

development = df.loc[df["official_split"].eq("train")].copy()
test_official = df.loc[df["official_split"].eq("test")].copy()
if HAS_OFFICIAL_TEST:
    heldout_text_keys = set(test_official["text_key"])
    before = len(development)
    development = development.loc[~development["text_key"].isin(heldout_text_keys)].copy()
    audit("train : recouvrement texte avec test officiel", before, len(development))
    test_official = clean_within(test_official, "text_key", "test officiel / texte")
development = clean_within(development, "text_key", "développement / texte")
display(pd.DataFrame(cleaning_log))
display(df[["text", "processed_text"]].sample(4, random_state=SEED))


## 8. Découpage stratifié puis mode rapide

Avec le test officiel, seule la partie de développement est divisée en train/validation
(90/10 par défaut). Sans test officiel, la branche alternative crée 80/10/10. Les doublons
de texte ayant été traités avant le découpage, une même entrée ne traverse pas les ensembles.

Le mode rapide prend un sous-ensemble **stratifié de chaque partition déjà séparée**. Le test
rapide est donc un sous-ensemble du test officiel, et non un test complet du corpus. Le mode
complet repart de toutes les données nettoyées. Une seconde vérification après tokenisation
retirera aussi les collisions dues à `[UNK]` et à la troncature.


In [ ]:
def stratified_split(frame, fraction):
    if frame["labels"].value_counts().min() < 2:
        raise ValueError("Chaque classe doit avoir au moins deux exemples pour stratifier.")
    left, right = train_test_split(frame, test_size=fraction, stratify=frame["labels"], random_state=SEED)
    return left.reset_index(drop=True), right.reset_index(drop=True)

if HAS_OFFICIAL_TEST:
    train_df, val_df = stratified_split(development, VAL_FRACTION)
    test_df = test_official.reset_index(drop=True)
else:
    print("Aucun test officiel dans cette distribution : découpage stratifié 80/10/10.")
    remaining, test_df = stratified_split(development, TEST_FRACTION)
    train_df, val_df = stratified_split(remaining, VAL_FRACTION / (1 - TEST_FRACTION))
splits = {"train": train_df, "validation": val_df, "test": test_df}
full_split_sizes = {name: len(frame) for name, frame in splits.items()}

def stratified_cap(frame, limit):
    if len(frame) <= limit:
        return frame.reset_index(drop=True)
    selected, _ = train_test_split(frame, train_size=limit, stratify=frame["labels"], random_state=SEED)
    return selected.reset_index(drop=True)

if QUICK_MODE:
    splits = {name: stratified_cap(frame, QUICK_LIMITS[name]) for name, frame in splits.items()}

def assert_disjoint(parts, key):
    names = list(parts)
    for i, name in enumerate(names):
        assert not parts[name][key].duplicated().any(), f"Doublon interne : {name}/{key}"
        for other in names[i + 1:]:
            assert set(parts[name][key]).isdisjoint(parts[other][key]), f"Fuite : {name}/{other}/{key}"

assert_disjoint(splits, "text_key")
display(pd.DataFrame({name: frame["labels"].value_counts().reindex(LABEL_IDS, fill_value=0)
                      for name, frame in splits.items()}).rename(index=id2label))


## 9. Choisir `max_length` sur le train uniquement

Les longueurs sont mesurées avec **le tokenizer du modèle**, tokens spéciaux compris, sans
troncature. On retient le 95e percentile du train arrondi au multiple de 8 supérieur, limité
par la capacité du modèle et un plafond mémoire de 256. La proportion tronquée est affichée.
Ni le test ni la validation ne servent à fixer ce paramètre.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=MODEL_REVISION, use_fast=True)
train_texts = splits["train"]["processed_text"].tolist()
lengths = []
for start in range(0, len(train_texts), 512):
    batch = tokenizer(train_texts[start:start + 512], truncation=False, padding=False,
                      add_special_tokens=True, return_length=True)
    lengths.extend(batch["length"])
lengths = np.asarray(lengths, dtype=np.int64)
display(pd.Series(lengths, name="tokens_train").describe(percentiles=[.50, .90, .95, .99]).to_frame())
model_limit = min(int(tokenizer.model_max_length), 512)
q_length = int(np.quantile(lengths, LENGTH_QUANTILE, method="higher"))
MAX_LENGTH = min(model_limit, MEMORY_LENGTH_CAP, max(16, 8 * math.ceil(q_length / 8)))
print("max_length :", MAX_LENGTH, "| part du train tronquée :", f"{(lengths > MAX_LENGTH).mean():.2%}")
plt.figure(figsize=(8, 3))
sns.histplot(lengths, bins=40)
plt.axvline(MAX_LENGTH, color="red", linestyle="--", label=f"max_length = {MAX_LENGTH}")
plt.xlabel("Tokens, tokens spéciaux compris")
plt.title("Longueurs sur le train uniquement")
plt.legend()
plt.tight_layout()
plt.savefig(RUN_DIR / "train_token_lengths.png", dpi=150)
plt.show()
for text in ["أنا سعيد 😊", "أنا حزين 😞", "لا أحب هذا"]:
    processed = preprocess_text(text)
    print(processed, "→", tokenizer.tokenize(processed))


## 10. Tokenisation, collisions et padding dynamique

`padding=False` stocke uniquement les tokens utiles. Le collateur complètera chaque batch à
sa longueur maximale. On compare ensuite les séquences `input_ids` **effectivement présentées
au modèle** : deux textes différents peuvent devenir identiques après normalisation ou troncature.

Les copies sont retirées de la partition de priorité inférieure : test, puis validation,
puis train. Cette décision utilise les entrées seules ; les contradictions d’étiquettes sont
traitées uniquement à l’intérieur de chaque partition. Toutes les suppressions sont comptées.
Ce contrôle couvre les doublons exacts d’entrée ; les paraphrases et quasi-doublons demanderaient
un audit sémantique complémentaire.


In [ ]:
tokenized_frames = {}
reserved_model_keys = set()
for name in ("test", "validation", "train"):
    frame = splits[name].copy()
    encoded = tokenizer(frame["processed_text"].tolist(), truncation=True, max_length=MAX_LENGTH, padding=False)
    for column, values in encoded.items():
        frame[column] = values
    frame["model_key"] = frame["input_ids"].map(lambda ids: tuple(ids))
    # Réserver les clés avant nettoyage : même un groupe exclu du test ne passe jamais au train.
    original_keys = set(frame["model_key"])
    before = len(frame)
    frame = frame.loc[~frame["model_key"].isin(reserved_model_keys)].copy()
    audit(f"{name} : recouvrement des séquences tokenisées", before, len(frame))
    frame = clean_within(frame, "model_key", f"{name} / tokens")
    tokenized_frames[name] = frame
    reserved_model_keys.update(original_keys)
splits = {name: tokenized_frames[name].reset_index(drop=True) for name in ("train", "validation", "test")}
assert_disjoint(splits, "text_key")
assert_disjoint(splits, "model_key")
for name, frame in splits.items():
    assert set(frame["labels"]) == set(LABEL_IDS), f"{name} a perdu une classe : augmenter les tailles rapides."
    assert frame["input_ids"].map(len).max() <= MAX_LENGTH
    if HAS_OFFICIAL_TEST and name == "test":
        assert frame["official_split"].eq("test").all()

model_columns = [key for key in tokenizer.model_input_names if key in splits["train"].columns] + ["labels"]
tokenized = DatasetDict({name: Dataset.from_pandas(frame[model_columns], preserve_index=False)
                         for name, frame in splits.items()})
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, padding=True,
                                       pad_to_multiple_of=8 if CUDA else None, return_tensors="pt")
demo_batch = data_collator([tokenized["train"][i] for i in range(min(3, len(tokenized["train"])) )])
print(tokenized)
print("Formes d’un batch :", {key: tuple(value.shape) for key, value in demo_batch.items()})
display(pd.DataFrame(cleaning_log))
pd.DataFrame(cleaning_log).to_csv(RUN_DIR / "cleaning_audit.csv", index=False, encoding="utf-8-sig")
for name, frame in splits.items():
    frame[["example_id", "source_file", "source_line", "text_key", "labels"]].to_csv(
        RUN_DIR / f"{name}_manifest.csv", index=False, encoding="utf-8-sig")


## 11. Modèle et métriques de validation

`AutoModelForSequenceClassification` charge les poids préentraînés et initialise une tête à deux
sorties. L’avertissement sur cette nouvelle tête est attendu : elle apprend pendant le fine-tuning.
**Tous les paramètres sont entraînables.** Aucun résultat de classification n’est pertinent avant
cet apprentissage.

Le F1 macro donne le même poids aux deux classes et sert à choisir le checkpoint. Accuracy,
précision et rappel macro sont calculés en complément.


In [ ]:
def metrics_from_arrays(y_true, y_pred):
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=LABEL_IDS, average="macro", zero_division=0)
    return {"accuracy": float(accuracy_score(y_true, y_pred)), "precision_macro": float(precision),
            "recall_macro": float(recall), "f1_macro": float(f1)}

def compute_metrics(eval_prediction):
    logits = eval_prediction.predictions
    if isinstance(logits, tuple):
        logits = logits[0]
    return metrics_from_arrays(eval_prediction.label_ids, np.argmax(logits, axis=-1))

set_seed(SEED)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, revision=MODEL_REVISION, num_labels=len(LABEL_IDS),
    id2label=id2label, label2id=label2id, use_safetensors=True)
model.requires_grad_(True)
total_parameters = sum(p.numel() for p in model.parameters())
trainable_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)
assert total_parameters == trainable_parameters
print(f"Paramètres entraînables : {trainable_parameters:,} / {total_parameters:,}")


## 12. Paramètres pour un GPU limité

En mode complet : taux d’apprentissage `2e-5`, **3 époques**, `weight_decay=0.01`, batch 8
sur les GPU de moins de 20 Gio, sinon 16. Une accumulation de gradients vise un batch effectif
d’au moins 16 sur un seul GPU. Le mode rapide utilise une époque et garde le vrai modèle.

BF16 n’est activé que sur un GPU CUDA qui le supporte nativement ; sinon FP16 est réservé aux
GPU CUDA de capacité suffisante et validé par un petit calcul. CPU et MPS restent en float32.
En cas de manque de mémoire, diminuer le batch et augmenter l’accumulation ; réduire ensuite
le plafond de longueur si nécessaire, en relançant les étapes de tokenisation.

L’API épinglée utilise `eval_strategy` et `processing_class`. La validation et la sauvegarde
ont lieu à chaque époque ; `load_best_model_at_end` recharge le meilleur F1 macro. Le test
n’est pas transmis au Trainer pour la sélection.


In [ ]:
gpu_memory_gib = min(torch.cuda.get_device_properties(i).total_memory / 2**30
                     for i in range(torch.cuda.device_count())) if CUDA else 0
BATCH_SIZE = 16 if CUDA and gpu_memory_gib >= 20 else 8
if DEVICE.type == "cpu":
    BATCH_SIZE = 2  # Repli CPU ; 8/16 sont les valeurs prévues sur GPU.
gpu_count = max(1, torch.cuda.device_count()) if CUDA else 1
GRADIENT_ACCUMULATION = max(1, math.ceil(16 / (BATCH_SIZE * gpu_count)))
USE_BF16 = False
USE_FP16 = False
if CUDA:
    native_bf16 = all(torch.cuda.get_device_capability(i)[0] >= 8 for i in range(torch.cuda.device_count()))
    fp16_capable = all(torch.cuda.get_device_capability(i) >= (5, 3) for i in range(torch.cuda.device_count()))
    candidate_dtype = (torch.bfloat16 if native_bf16 and torch.cuda.is_bf16_supported()
                       else torch.float16 if fp16_capable else None)
    if candidate_dtype is not None:
        try:
            with torch.autocast("cuda", dtype=candidate_dtype):
                probe = torch.ones((8, 8), device="cuda") @ torch.ones((8, 8), device="cuda")
            assert torch.isfinite(probe).all()
            USE_BF16 = candidate_dtype == torch.bfloat16
            USE_FP16 = candidate_dtype == torch.float16
            del probe
        except (RuntimeError, AssertionError):
            print("Calcul mixte non validé ; maintien en float32.")
print({"batch_par_périphérique": BATCH_SIZE, "accumulation": GRADIENT_ACCUMULATION,
       "batch_effectif": BATCH_SIZE * GRADIENT_ACCUMULATION * gpu_count,
       "bf16": USE_BF16, "fp16": USE_FP16, "époques": EPOCHS})

training_args = TrainingArguments(
    output_dir=str(RUN_DIR / "checkpoints"),
    learning_rate=2e-5, per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE, gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    num_train_epochs=EPOCHS, weight_decay=0.01, warmup_ratio=0.10,
    lr_scheduler_type="linear", optim="adamw_torch",
    eval_strategy="epoch", save_strategy="epoch", logging_strategy="steps",
    logging_steps=1 if QUICK_MODE else 50, logging_first_step=True,
    load_best_model_at_end=True, metric_for_best_model="f1_macro", greater_is_better=True,
    save_total_limit=2, save_safetensors=True,
    fp16=USE_FP16, bf16=USE_BF16, tf32=False,
    report_to=[], push_to_hub=False,
    seed=SEED, data_seed=SEED, dataloader_num_workers=0,
    dataloader_pin_memory=CUDA, eval_accumulation_steps=8,
    gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
)
trainer = Trainer(model=model, args=training_args,
                  train_dataset=tokenized["train"], eval_dataset=tokenized["validation"],
                  processing_class=tokenizer, data_collator=data_collator, compute_metrics=compute_metrics)
print("Trainer utilisera :", training_args.device)


## 13. Lancer le fine-tuning

Cette cellule réalise réellement la rétropropagation sur les poids d’AraBERT. Elle peut durer
longtemps sans GPU. Le temps, les pertes et les métriques affichés proviennent uniquement
de l’exécution présente. La validation sélectionne le meilleur checkpoint à la fin.


In [ ]:
train_result = trainer.train()
assert trainer.state.best_model_checkpoint is not None, "Aucun checkpoint sélectionné."
print("Meilleur checkpoint :", trainer.state.best_model_checkpoint)
print("Meilleur F1 macro de validation :", trainer.state.best_metric)
trainer.save_metrics("train", train_result.metrics)
trainer.save_state()
print("Fine-tuning terminé : le Trainer contient les poids du meilleur checkpoint.")


## 14. Courbes de perte et validation du modèle sélectionné

La perte d’entraînement est une moyenne des pas de journalisation ; la perte de validation
est calculée à la fin de chaque époque. Ces mesures portent sur des données différentes et
ne sont donc pas directement interchangeables. Un écart croissant peut signaler du surapprentissage.


In [ ]:
history = pd.DataFrame(trainer.state.log_history)
history.to_json(RUN_DIR / "training_history.json", orient="records", indent=2)
train_loss_history = history.dropna(subset=["loss"])
val_loss_history = history.dropna(subset=["eval_loss"])
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(train_loss_history["epoch"], train_loss_history["loss"], label="Train (moyennes journalisées)")
ax.plot(val_loss_history["epoch"], val_loss_history["eval_loss"], marker="o", label="Validation")
ax.set(xlabel="Époque", ylabel="Cross-entropy", title="Pertes pendant le fine-tuning")
ax.legend()
fig.tight_layout()
fig.savefig(RUN_DIR / "loss_curves.png", dpi=150)
plt.show()
validation_metrics = trainer.evaluate(eval_dataset=tokenized["validation"], metric_key_prefix="validation")
display(pd.Series(validation_metrics, name="validation").to_frame())
trainer.save_metrics("validation", validation_metrics)


## 15. Évaluation finale du test

Le modèle a été sélectionné uniquement avec la validation. **C’est ici que le test est évalué
pour la première fois.** On ne modifie plus les hyperparamètres à partir de ses erreurs.
Pour une nouvelle comparaison de modèles, utiliser la validation et réserver un nouveau test
indépendant si les décisions ont déjà été influencées par ce test.

Le rapport détaille précision, rappel, F1 et support par classe. La matrice utilise les lignes
pour les labels réels et les colonnes pour les labels prédits.


In [ ]:
assert trainer.state.best_model_checkpoint is not None
test_output = trainer.predict(tokenized["test"], metric_key_prefix="test")
test_logits = test_output.predictions[0] if isinstance(test_output.predictions, tuple) else test_output.predictions
test_probabilities = torch.softmax(torch.as_tensor(test_logits, dtype=torch.float32), dim=-1).numpy()
y_test = test_output.label_ids
y_pred = test_probabilities.argmax(axis=-1)
test_metrics = metrics_from_arrays(y_test, y_pred)
display(pd.Series(test_metrics, name="test").to_frame())
report_text = classification_report(y_test, y_pred, labels=LABEL_IDS, target_names=LABEL_NAMES,
                                    digits=4, zero_division=0)
print(report_text)
report_dict = classification_report(y_test, y_pred, labels=LABEL_IDS, target_names=LABEL_NAMES,
                                    output_dict=True, zero_division=0)
per_class = pd.DataFrame(report_dict).T.loc[LABEL_NAMES, ["precision", "recall", "f1-score", "support"]]
display(per_class)
cm = confusion_matrix(y_test, y_pred, labels=LABEL_IDS)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=LABEL_NAMES,
            yticklabels=LABEL_NAMES, ax=ax)
ax.set(xlabel="Label prédit", ylabel="Label réel", title="Matrice de confusion — test")
fig.tight_layout()
fig.savefig(RUN_DIR / "confusion_matrix.png", dpi=150)
plt.show()
trainer.save_metrics("test", {**test_output.metrics, **test_metrics})
(RUN_DIR / "classification_report.txt").write_text(report_text, encoding="utf-8")
(RUN_DIR / "classification_report.json").write_text(json.dumps(report_dict, ensure_ascii=False, indent=2), encoding="utf-8")


## 16. Examiner les erreurs

Le score affiché est la probabilité softmax associée à la classe prédite. On présente des
erreurs à score élevé et un échantillon aléatoire d’erreurs pour varier l’inspection. Ces exemples
servent au bilan qualitatif ; ils ne sont pas réinjectés dans l’entraînement de cette expérience.


In [ ]:
predictions_df = splits["test"][["example_id", "text", "processed_text", "labels"]].copy()
predictions_df["label_réel"] = predictions_df["labels"].map(id2label)
predictions_df["label_prédit"] = pd.Series(y_pred, index=predictions_df.index).map(id2label)
predictions_df["score_prédit"] = test_probabilities.max(axis=1)
predictions_df["prob_négatif"] = test_probabilities[:, 0]
predictions_df["prob_positif"] = test_probabilities[:, 1]
errors_df = predictions_df.loc[y_test != y_pred].copy()
print(f"Erreurs observées : {len(errors_df)} / {len(predictions_df)}")
columns = ["text", "label_réel", "label_prédit", "score_prédit"]
if errors_df.empty:
    print("Aucune erreur sur ce test ; aucune erreur artificielle n’est créée.")
else:
    most_confident = errors_df.nlargest(min(5, len(errors_df)), "score_prédit")
    remaining_errors = errors_df.drop(most_confident.index)
    varied = remaining_errors.sample(min(5, len(remaining_errors)), random_state=SEED)
    display(pd.concat([most_confident, varied])[columns])
predictions_df.to_csv(RUN_DIR / "test_predictions.csv", index=False, encoding="utf-8-sig")
errors_df.to_csv(RUN_DIR / "test_errors.csv", index=False, encoding="utf-8-sig")


## 17. Prédire de nouveaux textes

La fonction reprend exactement le prétraitement, le tokenizer et la longueur de l’entraînement.
Elle retourne la classe choisie et les probabilités des deux classes. **Une probabilité n’est
pas une certitude**, et ces scores ne sont pas calibrés. Un texte ambigu ou neutre reçoit tout
de même l’une des deux polarités car aucune classe neutre n’a été apprise.

Les exemples suivants sont des démonstrations, pas des résultats attendus. Les dialectes,
notamment la **darija marocaine**, les mélanges de langues, l’ironie et le sarcasme peuvent
être moins bien reconnus.


In [ ]:
inference_model = trainer.model
inference_model.eval()

def predict_sentiment(text):
    processed = preprocess_text(text)
    if not processed:
        raise ValueError("Le texte est vide après prétraitement.")
    encoded = tokenizer(processed, truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
    device = next(inference_model.parameters()).device
    encoded = {key: value.to(device) for key, value in encoded.items()}
    with torch.inference_mode():
        logits = inference_model(**encoded).logits
        probabilities = torch.softmax(logits.float(), dim=-1)[0].cpu().numpy()
    return {"sentiment": id2label[int(probabilities.argmax())],
            "probabilités": {id2label[i]: float(probabilities[i]) for i in LABEL_IDS}}

examples = ["الخدمة رائعة وأنا سعيد جداً 😊", "أحب هذا المنتج وأنصح به ❤️",
            "التجربة سيئة ولن أكررها 😞", "لا أحب هذه الخدمة أبداً",
            "الفيلم جيد لكن النهاية لم تعجبني", "وصل الطلب اليوم",
            "يا سلام، تأخر القطار ساعتين!", "هاد الخدمة زوينة بزاف 😊",
            "ما عجبنيش هاد المنتوج، خسارة الفلوس"]
display(pd.DataFrame([{"texte": text, **predict_sentiment(text)} for text in examples]))


## 18. Sauvegarder le modèle, le tokenizer et le contexte d’inférence

Les poids sauvegardés sont ceux du meilleur checkpoint de validation. Le tokenizer et les
paramètres de prétraitement sont indispensables pour reproduire les prédictions. On conserve
aussi les versions installées, les partitions, les métriques et les journaux de nettoyage.
Ces fichiers restent locaux ; rien n’est publié sur Hugging Face.


In [ ]:
EXPORT_DIR = RUN_DIR / "arabert_sentiment_model"
trainer.save_model(str(EXPORT_DIR))
tokenizer.save_pretrained(EXPORT_DIR)
inference_config = {"model_name": MODEL_NAME, "model_revision": MODEL_REVISION,
                    "max_length": int(MAX_LENGTH), "preprocessing": PREPROCESS_KWARGS,
                    "id2label": id2label, "label2id": label2id, "versions": VERSIONS}
(EXPORT_DIR / "inference_config.json").write_text(
    json.dumps(inference_config, ensure_ascii=False, indent=2), encoding="utf-8")
experiment_summary = {"mode": "rapide" if QUICK_MODE else "complet", "seed": SEED,
                      "official_test": HAS_OFFICIAL_TEST,
                      "raw_rows": len(raw_df), "full_split_sizes_before_token_cleanup": full_split_sizes,
                      "used_split_sizes": {name: len(frame) for name, frame in splits.items()},
                      "max_length": MAX_LENGTH, "epochs": EPOCHS,
                      "best_checkpoint": trainer.state.best_model_checkpoint,
                      "best_validation_f1_macro": float(trainer.state.best_metric),
                      "validation": validation_metrics, "test": test_metrics,
                      "versions": VERSIONS}
(RUN_DIR / "experiment_summary.json").write_text(
    json.dumps(experiment_summary, ensure_ascii=False, indent=2), encoding="utf-8")
print("Modèle sauvegardé dans :", EXPORT_DIR.resolve())
print("Poids :", [path.name for path in EXPORT_DIR.glob("*.safetensors")])


## 19. Recharger et vérifier une nouvelle prédiction

Cette cellule est autonome pour l’inférence après installation des dépendances : dans une
nouvelle session, définir `RELOAD_DIR` avec le chemin du dossier sauvegardé. Aucun entraînement
supplémentaire ni accès Internet n’est nécessaire pour recharger les poids et le tokenizer.
On compare aussi les prédictions avant/après sauvegarde avec une tolérance numérique.


In [ ]:
import os
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
import json
from pathlib import Path
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from arabert.preprocess import ArabertPreprocessor

# Dans une nouvelle session : remplacer la valeur par Path("chemin/arabert_sentiment_model").
RELOAD_DIR = Path(EXPORT_DIR) if "EXPORT_DIR" in globals() else Path("./arabert_sentiment_model")
reload_settings = json.loads((RELOAD_DIR / "inference_config.json").read_text(encoding="utf-8"))
reloaded_tokenizer = AutoTokenizer.from_pretrained(RELOAD_DIR, local_files_only=True)
# En exécution continue, garder cette copie sur CPU évite de doubler les poids en VRAM.
reload_device = torch.device("cpu" if "trainer" in globals() else "cuda" if torch.cuda.is_available() else "cpu")
reloaded_model = AutoModelForSequenceClassification.from_pretrained(
    RELOAD_DIR, local_files_only=True, use_safetensors=True).to(reload_device).eval()
reloaded_preprocessor = ArabertPreprocessor(**reload_settings["preprocessing"])
reloaded_labels = {int(key): value for key, value in reloaded_model.config.id2label.items()}

def predict_sentiment_reloaded(text):
    if not isinstance(text, str):
        raise TypeError("Le texte doit être une chaîne Unicode.")
    processed = reloaded_preprocessor.preprocess(text).strip()
    if not processed:
        raise ValueError("Le texte est vide après prétraitement.")
    inputs = reloaded_tokenizer(processed, truncation=True, max_length=reload_settings["max_length"],
                                return_tensors="pt").to(reload_device)
    with torch.inference_mode():
        probabilities = torch.softmax(reloaded_model(**inputs).logits.float(), dim=-1)[0].cpu().tolist()
    predicted_id = max(range(len(probabilities)), key=probabilities.__getitem__)
    return {"sentiment": reloaded_labels[predicted_id],
            "probabilités": {reloaded_labels[i]: float(p) for i, p in enumerate(probabilities)}}

reload_example = "أنا سعيد بهذه التجربة 😊"
reloaded_prediction = predict_sentiment_reloaded(reload_example)
print(reloaded_prediction)
if "predict_sentiment" in globals():
    original_prediction = predict_sentiment(reload_example)
    before_prob = torch.tensor(list(original_prediction["probabilités"].values()))
    after_prob = torch.tensor(list(reloaded_prediction["probabilités"].values()))
    assert torch.allclose(before_prob, after_prob, atol=1e-4, rtol=1e-4), "Vérifier l’identité des poids/paramètres."
    print("Cohérence avant/après sauvegarde vérifiée (tolérance 1e-4).")


## 20. Conclusion fondée sur les résultats de cette exécution

Le bilan ci-dessous est calculé à partir des mesures réellement obtenues. Sans exécution de
l’entraînement, aucune accuracy ni aucun F1 ne peut être annoncé. Les résultats dépendent du
nettoyage et portent sur un test dédupliqué : ils ne sont pas directement comparables à un score
publié sur le corpus brut.

**Limites :** le sarcasme demande du contexte ; les dialectes, la darija et l’Arabizi peuvent
être sous-représentés ; les biais des utilisateurs et thèmes de Twitter limitent la généralisation.
L’auteur indique une collecte fondée sur un lexique d’emojis positifs et négatifs : cette annotation
distante peut être bruitée et le modèle peut exploiter ces indices plutôt que comprendre le texte.
Les emojis conservés rendent ce point particulièrement important. Deux polarités ne couvrent
ni la neutralité ni les émotions mixtes. L’absence d’identifiants d’auteur limite les contrôles
de fuite entre utilisateurs ; les assertions garantissent seulement la séparation des doublons
exacts de texte et d’entrées tokenisées.

**Améliorations réalistes :** comparer le modèle Twitter d’AraBERT avec le même protocole de
validation ; enrichir et réannoter un jeu en darija ; évaluer la robustesse avec et sans emojis ;
essayer plusieurs graines et intervalles de confiance ; annoter un jeu indépendant avec neutre ;
rechercher les quasi-doublons ; calibrer les probabilités sur la validation et ajouter une option
d’abstention. Régler longueur, taux d’apprentissage et régularisation uniquement sur la validation.


In [ ]:
mode_description = "diagnostic sur sous-ensembles" if QUICK_MODE else "expérience complète sur corpus nettoyé"
sizes = {name: len(frame) for name, frame in splits.items()}
conclusion = f"""### Résultats mesurés — {mode_description}

- Données utilisées : **{sizes['train']} train**, **{sizes['validation']} validation**, **{sizes['test']} test**.
- Fine-tuning : **{EPOCHS} époque(s)** ; longueur maximale : **{MAX_LENGTH} tokens**.
- Meilleur F1 macro de validation pendant la sélection : **{trainer.state.best_metric:.4f}**.
- Test final : accuracy **{test_metrics['accuracy']:.4f}**, précision macro **{test_metrics['precision_macro']:.4f}**,
  rappel macro **{test_metrics['recall_macro']:.4f}**, F1 macro **{test_metrics['f1_macro']:.4f}**.
- Erreurs observées : **{len(errors_df)} / {sizes['test']}**.
- Modèle exporté : `{EXPORT_DIR.resolve()}`.
"""
if QUICK_MODE:
    conclusion += "\n**Mode rapide : ces mesures ne constituent pas une évaluation du corpus complet.**\n"
display(Markdown(conclusion))
(RUN_DIR / "conclusion.md").write_text(conclusion, encoding="utf-8")
